In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split                                              
import joblib
import scipy.stats as stats

dataset = "medical"
MODEL_PATH = f"../models/{dataset}/xgboost_multi_output.joblib"
DATA_PATH = f"../data/processed/proc_{dataset}.csv"
N_STATISTICS = 1000  
N_VISUAL = 20        


print(f"Iniciando validação para: {dataset}")

# Carregar dados
model = joblib.load(MODEL_PATH)
df = pd.read_csv(DATA_PATH)
print(f"Dataset carregado: {df.shape[0]} linhas")
print(f"Modelo carregado")

# Pegando features e os targets
try:
    X = df.drop(columns=['F1 (macro averaged by label)', 'Model Size', 'Model Size Log'])
    y = df[['F1 (macro averaged by label)', 'Model Size Log']]
except KeyError as e:
    print(f"ERRO: Colunas não encontradas no CSV. Verifique os nomes exatos.\nErro: {e}")

# Recuperando os dados de teste que o modelo não foi treinado
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inferência em Massa 
print("Realizando predição no test set")
preds = model.predict(X_test)

# Extrair apenas a coluna de F1 
if preds.ndim > 1:
    y_pred_f1 = preds[:, 0]
else:
    y_pred_f1 = preds

# Valores reais de F1 do y_test
y_real_f1 = y_test.iloc[:, 0].values
    
# DataFrame para análise
df_results = pd.DataFrame({
    'Real_F1': y_real_f1,
    'Pred_F1': y_pred_f1
})

#------------------------------------------------------------------------------------------------
# Teste de correlação de Spearman
# Objetivo: avaliar se o modelo sabe predizer 
# o F1 dos pipelines que nunca viu

# Spearman em todo o conjunto
spearman_all, _ = stats.spearmanr(df_results['Real_F1'], df_results['Pred_F1'])
limit = int(len(df_results) * 0.10)

# Spearman nos melhores 10% pipelines
df_filtered = df_results.sort_values(by='Real_F1', ascending=False).head(limit)
spearman_filtered, _ = stats.spearmanr(df_filtered['Real_F1'], df_filtered['Pred_F1'])

print(f"Pipelines avaliados: {len(df_results)}")
print(f"Spearman Global: {spearman_all:.4f}")
print(f"Spearman nos 10%: {spearman_filtered:.4f}")

# Perda de Performance:

# Melhor pipeline
best_real = df_results['Real_F1'].max()
# Melhor pipeline predito
idx_best_predicted = df_results['Pred_F1'].idxmax()
# F1 real do pipeline predito
real_f1 = df_results.loc[idx_best_predicted, 'Real_F1']

loss = best_real - real_f1
print(f"Melhor F1 Disponível: {best_real:.3f}")
print(f"F1 do Pipeline Escolhido pelo Modelo: {real_f1:.3f}")
print(f"Perda de Performance (Regret): {loss:.3f}")

Iniciando validação para: medical
Dataset carregado: 73642 linhas
Modelo carregado
Realizando predição no test set
Pipelines avaliados: 14729
Spearman Global: 0.9857
Spearman nos 10%: 0.6515
Melhor F1 Disponível: 0.809
F1 do Pipeline Escolhido pelo Modelo: 0.791
Perda de Performance (Regret): 0.018
